In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import pandas as pd
df = pd.read_csv('/content/adult_mental_health.csv', encoding='latin1',on_bad_lines='skip',engine='python')
df

,Age,Prompt,Label
0,41,Feeling relaxed and calm,Depression
1,40,I don't enjoy things like before,Depression
2,34,Work is exhausting me mentally,Stress
3,26,I feel like ending everything,Suicidal
4,23,Work is exhausting me mentally,Stress
...,...,...,...
54995,42,Deadlines are stressing me out,Stress
54996,32,I feel like ending everything,Suicidal
54997,18,Nothing excites me anymore,Depression
54998,29,I can't take this pain anymore,Stress


In [ ]:
from textblob import TextBlob #Imports TextBlob library (used for sentiment analysis).
import re
!pip install emoji # Install the emoji library
import emoji #Imports re (for regex text cleaning) and emoji (to convert emojis into words).

df = pd.read_csv("/content/adult_mental_health.csv", encoding='latin1', on_bad_lines='skip', engine='python')
df = df.rename(columns={'Prompt': 'statement', 'Label': 'status'})

stress_keywords = ["depressed", "hopeless", "tired", "stressed", "anxious"] #Creates a list of keywords that often appear in stressful or depressed posts.

def preprocess(text):
    text = re.sub(r"http\S+|@\S+|#\S+", "", text) # Remove the all HTTP URL.Only accept URL Before or After text
    text = emoji.demojize(text) #Converts emojis into text form.
    return text.lower().strip() #Converts text to lowercase.Removes spaces from start/end.Returns cleaned text.

def detect_stress(text): #Defines a function detect_stress that checks if text has stress keywords.
    return any(word in text for word in stress_keywords) #Loops through stress_keywords.If any word exists in text, returns True. Otherwise False.

def analyze_sentiment(text): #Defines a function analyze_sentiment that checks if text is Positive, Negative, or Neutral.
    sentiment = TextBlob(text).sentiment.polarity #Uses TextBlob to calculate polarity (sentiment score).Range: -1.0 (very negative) → +1.0 (very positive).
    if sentiment < -0.1: return "Negative" #If polarity is less than -0.1  label as Negative.
    elif sentiment > 0.1: return "Positive"#If polarity is greater than 0.1  label as Positive.
    else: return "Neutral" #If polarity is between -0.1 and +0.1 label as Neutral.

def compute_risk(stress, sentiment, ratio, th_ratio=0.5):
    score = 0 #Start with a base risk score of 0.
    if stress: score += 1 #If stress keywords are present add 1 point to score.
    if sentiment == "Negative": score += 1 #If sentiment is negative  add 1 point.
    if ratio < th_ratio: score += 1 #If ratio of positives/negatives is too low add 1 point.
    return score

posts = ["I am feeling so depressed today 😔", "I love my child but it's exhausting."]
processed = [preprocess(p) for p in posts] #Applies preprocess function to each post.

neg_count, pos_count = 0, 0 #Initialize counters for negative and positive posts.
for p in processed:
    stress = detect_stress(p) #Check if post contains stress keywords.
    sentiment = analyze_sentiment(p) #Analyze sentiment (Negative/Positive/Neutral).
    if sentiment == "Negative": neg_count += 1 #If sentiment is negative, increase neg_count by 1.
    if sentiment == "Positive": pos_count += 1 #If sentiment is positive, increase pos_count by 1.
    ratio = pos_count / (neg_count+1) #Compute positive-to-negative ratio.
    risk = compute_risk(stress, sentiment, ratio) #Calculate risk score for this post.
    print(p, "=>", sentiment, "| Risk Score:", risk) #Print post, its sentiment, and the calculated risk score.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 10.1 MB/s eta 0:00:00
i am feeling so depressed today :pensive_face: => Neutral | Risk Score: 2
i love my child but it's exhausting. => Neutral | Risk Score: 1


In [ ]:
# 1. Import Required Libraries
import pandas as pd
import numpy as np
import re, emoji
import nltk
from nltk.corpus import stopwords
from textblob import TextBlob
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# 2. Download Stopwords from NLTK
nltk.download("stopwords")
stop_words = set(stopwords.words("english"))

# 3. Load Dataset
df = pd.read_csv("/content/adult_mental_health.csv", encoding='latin1', on_bad_lines='skip', engine='python')
df

# Rename columns to 'statement' and 'status' for consistency with the rest of the script.
# The actual columns in adult_mental_health.csv are 'ï»¿Age', 'Prompt', 'Label'.
# We will rename 'Prompt' to 'statement' and 'Label' to 'status'.
# The 'ï»¿Age' column is left as is, as it's not causing an issue and is not explicitly dropped or used as 'statement'/'status'.
df = df.rename(columns={'Prompt': 'statement', 'Label': 'status'})

# Remove missing values from key columns
df = df.dropna(subset=["statement"])
df = df.dropna(subset=["status"])

# Print dataset info
print("Dataset shape:", df.shape)
print(df.head())
print("Value counts of 'status' before preprocessing:", df['status'].value_counts())

# 4. Preprocessing Function
def preprocess(text):
    text = text.lower()  # convert to lowercase
    text = re.sub(r"http\S+|www\S+|@\S+|#\S+", "", text)  # remove links, mentions, hashtags
    text = emoji.demojize(text)  # convert emojis to text
    text = re.sub(r"[^a-z\s]", "", text)  # keep only alphabets
    text = " ".join([w for w in text.split() if w not in stop_words])  # remove stopwords
    return text

# Apply preprocessing to dataset
df["clean_text"] = df["statement"].astype(str).apply(preprocess)

# Print after preprocessing
print("\nValue counts of 'status' after preprocessing:", df['status'].value_counts())
print("\nHead of dataframe after preprocessing:")
print(df.head())

# 5. Rule-Based Stress + Sentiment
# Keywords for stress detection
stress_keywords = ["depressed", "hopeless", "tired", "stressed", "anxious", "suicidal", "sad", "angry"]

# Detect stress if any keyword is present
def detect_stress(text):
    return any(word in text for stress_keyword in stress_keywords for word in text.split() if stress_keyword in word)

# Sentiment analysis using TextBlob
def analyze_sentiment(text):
    sentiment = TextBlob(text).sentiment.polarity
    if sentiment < -0.1: return "Negative"
    elif sentiment > 0.1: return "Positive"
    else: return "Neutral"

# Compute risk score (rule-based)
def compute_risk(stress, sentiment, ratio, th_ratio=0.5):
    score = 0
    if stress: score += 1
    if sentiment == "Negative": score += 1
    if ratio < th_ratio: score += 1
    return score

# Rule-based demo with first 10 samples
print("\n--- Rule-Based Demo ---")
neg_count, pos_count = 0, 0
for i, row in df.head(10).iterrows():
    text = row["clean_text"]
    stress = detect_stress(text)
    sentiment = analyze_sentiment(text)
    if sentiment == "Negative": neg_count += 1
    if sentiment == "Positive": pos_count += 1
    ratio = pos_count / (neg_count + 1)  # avoid divide by zero
    risk = compute_risk(stress, sentiment, ratio)
    print(f"{text[:50]}... | Sentiment={sentiment} | Stress={stress} | Risk={risk}")

# 6. Prepare Data for ML
X = df["clean_text"]  # features
y = df["status"]      # target labels

print("\nShape of X:", X.shape)
print("Shape of y:", y.shape)
print("Value counts of y before splitting:", y.value_counts())

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("\nShape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_test:", y_test.shape)
print("Value counts of y_train after splitting:", y_train.value_counts())
print("Value counts of y_test after splitting:", y_test.value_counts())

# 7. Text Vectorization (TF-IDF)
tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("\nShape of X_train_tfidf:", X_train_tfidf.shape)
print("Shape of X_test_tfidf:", X_test_tfidf.shape)

# Sanity checks
print("\nChecking for NaN in X_train_tfidf:", np.isnan(X_train_tfidf.data).any())
print("Checking for infinity in X_train_tfidf:", np.isinf(X_train_tfidf.data).any())
print("Checking for NaN in y_train:", y_train.isnull().any())

# 8. Train ML Model (Logistic Regression)
model = LogisticRegression(max_iter=500)
model.fit(X_train_tfidf, y_train)

# Predict on test set
y_pred = model.predict(X_test_tfidf)

# Show classification report
print("\n--- ML Classification Report ---")
print(classification_report(y_test, y_pred))
print("Accuracy:", accuracy_score(y_test, y_pred))

# 9. Unified Prediction Function
def predict_post(text):
    clean = preprocess(text)              # preprocess text
    vec = tfidf.transform([clean])        # vectorize text
    ml_pred = model.predict(vec)[0]       # ML prediction

    stress = detect_stress(clean)         # rule-based stress
    sentiment = analyze_sentiment(clean)  # rule-based sentiment
    rb_score = compute_risk(stress, sentiment, 0.5)  # risk score

    return {
        "text": text,
        "rule_based_sentiment": sentiment,
        "rule_based_risk": rb_score,
        "ml_prediction": ml_pred
    }

# Demo prediction
example = "I feel so tired and hopeless these days \ud83d\ude22"
print("\n--- Demo Prediction ---")
print(predict_post(example))


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


Dataset shape: (55000, 3)
   Age                         statement      status
0   41          Feeling relaxed and calm  Depression
1   40  I don't enjoy things like before  Depression
2   34    Work is exhausting me mentally      Stress
3   26     I feel like ending everything    Suicidal
4   23    Work is exhausting me mentally      Stress
Value counts of 'status' before preprocessing: status
Suicidal      13806
Stress        13786
Depression    13723
Normal        13685
Name: count, dtype: int64

Value counts of 'status' after preprocessing: status
Suicidal      13806
Stress        13786
Depression    13723
Normal        13685
Name: count, dtype: int64

Head of dataframe after preprocessing:
   Age                         statement      status  \
0   41          Feeling relaxed and calm  Depression   
1   40  I don't enjoy things like before  Depression   
2   34    Work is exhausting me mentally      Stress   
3   26     I feel like ending everything    Suicidal   
4   23    Work i

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
import pandas as pd
df = pd.read_csv('/content/child_mental_health.csv', encoding='latin1',on_bad_lines='skip',engine='python')
df

,Age,Prompt,Label
0,9,Bedtime just makes me want to cry all the time.,Depression
1,9,Recess would be better if I disappeared lately.,Depression
2,7,School would be better if I disappeared yester...,Suicidal
3,13,The dark is really scary sometimes.,Anxiety
4,13,Homework is really scary sometimes.,Anxiety
...,...,...,...
19995,8,School was fun today every morning.,Normal
19996,6,Bedtime would be better if I disappeared durin...,Suicidal
19997,9,"My math test is fine, I guess all the time.",Normal
19998,15,My bedroom is something I don't care about som...,Depression


In [ ]:
print(df.info())
print(df['Label'].value_counts())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Age     20000 non-null  int64 
 1   Prompt  20000 non-null  object
 2   Label   20000 non-null  object
dtypes: int64(1), object(2)
memory usage: 468.9+ KB
None
Label
Depression    5134
Anxiety       4973
Suicidal      4959
Normal        4934
Name: count, dtype: int64


In [ ]:
X = df['Prompt']      # Text input
y = df['Label']       # Target class

In [ ]:
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)

X_vectorized = vectorizer.fit_transform(X)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_vectorized, y, test_size=0.2, random_state=42
)

In [ ]:
model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
accuracy = accuracy_score(y_test, y_pred)

print("Model Accuracy:", accuracy)

Model Accuracy: 0.833


In [ ]:
print("Classification Report:\n")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))

Classification Report:

              precision    recall  f1-score   support

     Anxiety       0.83      0.82      0.82      1019
  Depression       0.83      0.83      0.83       999
      Normal       0.83      0.87      0.85       987
    Suicidal       0.84      0.81      0.83       995

    accuracy                           0.83      4000
   macro avg       0.83      0.83      0.83      4000
weighted avg       0.83      0.83      0.83      4000

Confusion Matrix:

[[835  67  63  54]
 [ 56 833  57  53]
 [ 48  36 859  44]
 [ 69  62  59 805]]


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [ ]:
# Load datasets
adult_df = pd.read_csv("/content/adult_mental_health.csv")
child_df = pd.read_csv("/content/child_mental_health.csv")
print(adult_df.head())
print(child_df.head())


   Age                            Prompt       Label
0   41          Feeling relaxed and calm  Depression
1   40  I don't enjoy things like before  Depression
2   34    Work is exhausting me mentally      Stress
3   26     I feel like ending everything    Suicidal
4   23    Work is exhausting me mentally      Stress
   Age                                             Prompt       Label
0    9   Bedtime just makes me want to cry all the time.   Depression
1    9   Recess would be better if I disappeared lately.   Depression
2    7  School would be better if I disappeared yester...    Suicidal
3   13               The dark is really scary sometimes.      Anxiety
4   13               Homework is really scary sometimes.      Anxiety


In [ ]:
# Combine labels (important to align classes)
all_labels = list(set(adult_df['Label']).union(set(child_df['Label'])))

In [ ]:
# Convert labels to same format
adult_df = adult_df[adult_df['Label'].isin(all_labels)]
child_df = child_df[child_df['Label'].isin(all_labels)]

In [ ]:
# TF-IDF (IMPORTANT: same vectorizer for both)
vectorizer = TfidfVectorizer(max_features=5000)

In [ ]:
adult_df['Prompt'] = adult_df['Prompt'].fillna('')
child_df['Prompt'] = child_df['Prompt'].fillna('')

In [ ]:
# The vectorizer must be fitted before transforming.
# Consolidating fit and transform steps here to ensure correct order.

# Ensure the vectorizer initialized in xmHATYJoOzHJ is used and fitted.
# If you intend to use a different vectorizer, initialize it here.

# Existing vectorizer from xmHATYJoOzHJ:
# vectorizer = TfidfVectorizer(max_features=5000) # This line is already in xmHATYJoOzHJ

vectorizer.fit(pd.concat([adult_df['Prompt'], child_df['Prompt']]))

X_adult = vectorizer.transform(adult_df['Prompt'])
y_adult = adult_df['Label']

X_child = vectorizer.transform(child_df['Prompt'])
y_child = child_df['Label']

In [ ]:
# Fill missing text
adult_df['Prompt'] = adult_df['Prompt'].fillna('')
child_df['Prompt'] = child_df['Prompt'].fillna('')

# Remove missing labels
adult_df = adult_df.dropna(subset=['Label'])
child_df = child_df.dropna(subset=['Label'])

In [ ]:
# Remove rows where Prompt OR Label is missing
adult_df = adult_df.dropna(subset=['Prompt', 'Label'])
child_df = child_df.dropna(subset=['Prompt', 'Label'])

# Remove empty strings
adult_df = adult_df[
    (adult_df['Prompt'].astype(str).str.strip() != '') &
    (adult_df['Label'].astype(str).str.strip() != '')
]

child_df = child_df[
    (child_df['Prompt'].astype(str).str.strip() != '') &
    (child_df['Label'].astype(str).str.strip() != '')
]

# Reset index
adult_df = adult_df.reset_index(drop=True)
child_df = child_df.reset_index(drop=True)

In [ ]:
# This cell's content for vectorizer fitting has been moved to E_wD-GRDpVG2 to ensure correct execution order.
# It is now commented out to prevent redundant operations.
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(max_features=5000)
vectorizer.fit(pd.concat([adult_df['Prompt'], child_df['Prompt']]))

TfidfVectorizer(max_features=5000)

In [ ]:
# This cell's content for vectorizer transformation has been moved to E_wD-GRDpVG2 to ensure correct execution order.
# It is now commented out to prevent redundant operations.
X_adult = vectorizer.transform(adult_df['Prompt'])
y_adult = adult_df['Label']
X_child = vectorizer.transform(child_df['Prompt'])
y_child = child_df['Label']

In [ ]:
import numpy as np

print("NaN in X_adult:", np.isnan(X_adult.toarray()).sum())
print("NaN in X_child:", np.isnan(X_child.toarray()).sum())

print("NaN in y_adult:", y_adult.isna().sum())
print("NaN in y_child:", y_child.isna().sum())

NaN in X_adult: 0
NaN in X_child: 0
NaN in y_adult: 0
NaN in y_child: 0


In [ ]:
y_adult = y_adult.fillna("Unknown")
y_child = y_child.fillna("Unknown")

In [ ]:
X_adult = np.nan_to_num(X_adult.toarray())
X_child = np.nan_to_num(X_child.toarray())

In [ ]:
from sklearn.linear_model import LogisticRegression

model_adult = LogisticRegression(max_iter=500)
model_child = LogisticRegression(max_iter=500)

model_adult.fit(X_adult, y_adult)
model_child.fit(X_child, y_child)

LogisticRegression(max_iter=500)

In [ ]:
# Train models
model_adult = LogisticRegression(max_iter=500)
model_child = LogisticRegression(max_iter=500)

model_adult.fit(X_adult, y_adult)
model_child.fit(X_child, y_child)

LogisticRegression(max_iter=500)

In [ ]:
# Find common labels
common_labels = set(adult_df['Label']).intersection(set(child_df['Label']))

# Filter both datasets
adult_df = adult_df[adult_df['Label'].isin(common_labels)]
child_df = child_df[child_df['Label'].isin(common_labels)]

In [ ]:
model_adult.fit(X_adult, y_adult)
model_child.fit(X_child, y_child)

LogisticRegression(max_iter=500)

In [ ]:
print(model_adult.classes_)
print(model_child.classes_)

['Depression' 'Normal' 'Stress' 'Suicidal']
['Anxiety' 'Depression' 'Normal' 'Suicidal']


In [ ]:
common_labels = list(
    set(adult_df['Label']).intersection(set(child_df['Label']))
)

print("Common Labels:", common_labels)

Common Labels: ['Depression', 'Suicidal', 'Normal']


In [ ]:
adult_df = adult_df[adult_df['Label'].isin(common_labels)]
child_df = child_df[child_df['Label'].isin(common_labels)]

In [ ]:
adult_df = adult_df.reset_index(drop=True)
child_df = child_df.reset_index(drop=True)

In [ ]:
# Refit vectorizer
vectorizer.fit(pd.concat([adult_df['Prompt'], child_df['Prompt']]))

# Transform
X_adult = vectorizer.transform(adult_df['Prompt'])
y_adult = adult_df['Label']

X_child = vectorizer.transform(child_df['Prompt'])
y_child = child_df['Label']

# Train again
model_adult.fit(X_adult, y_adult)
model_child.fit(X_child, y_child)

LogisticRegression(max_iter=500)

In [ ]:
print(model_adult.classes_)
print(model_child.classes_)

['Depression' 'Normal' 'Suicidal']
['Depression' 'Normal' 'Suicidal']


In [ ]:
# Sort labels before training
adult_df = adult_df.sort_values('Label')
child_df = child_df.sort_values('Label')

In [ ]:
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1,2))

In [ ]:
vectorizer.fit(pd.concat([adult_df['Prompt'], child_df['Prompt']]))

TfidfVectorizer(max_features=10000, ngram_range=(1, 2))

In [ ]:
# Re-transform X_adult and X_child with the newly fitted vectorizer
X_adult = vectorizer.transform(adult_df['Prompt'])
X_child = vectorizer.transform(child_df['Prompt'])

In [ ]:
model_adult.fit(X_adult, y_adult)
model_child.fit(X_child, y_child)

LogisticRegression(max_iter=500)

In [ ]:
import re
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

adult_df['Prompt'] = adult_df['Prompt'].apply(clean_text)
child_df['Prompt'] = child_df['Prompt'].apply(clean_text)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd # Import pandas here to ensure it's available

vectorizer = TfidfVectorizer(
    max_features=12000,
    ngram_range=(1,2),
    min_df=2,
    max_df=0.85,
    sublinear_tf=True
)

# Fit the vectorizer on the combined (cleaned) prompts
vectorizer.fit(pd.concat([adult_df['Prompt'], child_df['Prompt']]))

TfidfVectorizer(max_df=0.85, max_features=12000, min_df=2, ngram_range=(1, 2),
                sublinear_tf=True)

In [ ]:
from sklearn.linear_model import LogisticRegression

model_adult = LogisticRegression(max_iter=1500, C=3)
model_child = LogisticRegression(max_iter=1500, C=3)

In [ ]:
# Ensure X_adult and X_child are transformed with the current vectorizer (12000 features)
X_adult = vectorizer.transform(adult_df['Prompt'])
X_child = vectorizer.transform(child_df['Prompt'])

# Fit the models first, as they were re-initialized in a previous cell
model_adult.fit(X_adult, y_adult)
model_child.fit(X_child, y_child)

model1_adult = model_adult.score(X_adult, y_adult)
model2_child = model_child.score(X_child, y_child)

w1 = model1_adult / (model1_adult + model2_child)
w2 = model2_child / (model1_adult + model2_child)

new_coef = (w1 * model_adult.coef_ + w2 * model_child.coef_)
new_intercept = (w1 * model_adult.intercept_ + w2 * model_child.intercept_)

In [ ]:
from sklearn.linear_model import LogisticRegression

# Ensure the vectorizer is fitted immediately before use
vectorizer.fit(pd.concat([adult_df['Prompt'], child_df['Prompt']]))

X_test = vectorizer.transform(pd.concat([adult_df['Prompt'], child_df['Prompt']]))
y_test = pd.concat([adult_df['Label'], child_df['Label']])

# Create a new LogisticRegression model for the merged coefficients
merged_model = LogisticRegression(max_iter=1500, C=3) # Use the same parameters as model_adult/child

# Manually set the coefficients and intercept from the weighted average
# Ensure the number of features matches (X_test.shape[1] or vectorizer.get_feature_names_out().shape[0])
# The classes must also match the order used to compute new_coef and new_intercept
merged_model.classes_ = model_adult.classes_ # Assuming both models have the same class order after filtering
merged_model.coef_ = new_coef
merged_model.intercept_ = new_intercept

y_pred = merged_model.predict(X_test)

In [ ]:
new_coef = (model_adult.coef_ + model_child.coef_) / 2
new_intercept = (model_adult.intercept_ + model_child.intercept_) / 2

In [ ]:
print("Merged Model Classification Report:\n")
print(classification_report(y_test, y_pred))

Merged Model Classification Report:

              precision    recall  f1-score   support

  Depression       0.23      0.27      0.25     18857
      Normal       0.53      0.24      0.33     18619
    Suicidal       0.26      0.35      0.30     18765

    accuracy                           0.29     56241
   macro avg       0.34      0.29      0.29     56241
weighted avg       0.34      0.29      0.29     56241



In [ ]:
print("Merged Model Confusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))

Merged Model Confusion Matrix:

[[ 5180  2152 11525]
 [ 7158  4449  7012]
 [10267  1850  6648]]


In [ ]:
def predict_merged_model(text):
    # Clean the input text using the previously defined clean_text function
    cleaned_text = clean_text(text)

    # Vectorize the cleaned text using the fitted vectorizer
    text_vectorized = vectorizer.transform([cleaned_text])

    # Make prediction using the merged model
    prediction = merged_model.predict(text_vectorized)[0]

    return prediction

# Example usage of the prediction function
example_text_1 = "I feel so happy today, life is great!"
example_text_2 = "I can't cope with anything anymore, I just want it to end."
example_text_3 = "Sometimes I feel down, but mostly I am okay."

print(f"Prediction for \"{example_text_1}\": {predict_merged_model(example_text_1)}")
print(f"Prediction for \"{example_text_2}\": {predict_merged_model(example_text_2)}")
print(f"Prediction for \"{example_text_3}\": {predict_merged_model(example_text_3)}")

Prediction for "I feel so happy today, life is great!": Suicidal
Prediction for "I can't cope with anything anymore, I just want it to end.": Depression
Prediction for "Sometimes I feel down, but mostly I am okay.": Suicidal


In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd

# Combine the processed adult and child prompts and labels
# X_test_merged and y_test_merged already hold the combined and vectorized data
X_combined = vectorizer.transform(pd.concat([adult_df['Prompt'], child_df['Prompt']]))
y_combined = pd.concat([adult_df['Label'], child_df['Label']])

# Split the combined data into training and testing sets
X_train_tuned, X_test_tuned, y_train_tuned, y_test_tuned = train_test_split(
    X_combined, y_combined, test_size=0.2, random_state=42, stratify=y_combined
)

print("Shape of X_train_tuned:", X_train_tuned.shape)
print("Shape of X_test_tuned:", X_test_tuned.shape)
print("Value counts of y_train_tuned:\n", y_train_tuned.value_counts())
print("Value counts of y_test_tuned:\n", y_test_tuned.value_counts())

Shape of X_train_tuned: (44992, 457)
Shape of X_test_tuned: (11249, 457)
Value counts of y_train_tuned:
 Label
Depression    15085
Suicidal      15012
Normal        14895
Name: count, dtype: int64
Value counts of y_test_tuned:
 Label
Depression    3772
Suicidal      3753
Normal        3724
Name: count, dtype: int64


In [ ]:
# Define the Logistic Regression model
lr_model = LogisticRegression(max_iter=2000) # Increase max_iter for convergence

# Define the parameter grid to search
param_grid = {
    'C': [0.1, 1, 10, 100],  # Regularization strength
    'solver': ['liblinear', 'saga'] # Solvers that work well with L1/L2 regularization
}

# Initialize GridSearchCV
grid_search = GridSearchCV(estimator=lr_model, param_grid=param_grid, cv=5, n_jobs=-1, verbose=2, scoring='accuracy')

# Fit GridSearchCV to the training data
grid_search.fit(X_train_tuned, y_train_tuned)

# Print the best parameters and best score
print("\nBest parameters found:", grid_search.best_params_)
print("Best cross-validation accuracy:", grid_search.best_score_)

# Get the best estimator
best_lr_model = grid_search.best_estimator_

# Make predictions on the test set with the best model
y_pred_tuned = best_lr_model.predict(X_test_tuned)

# Evaluate the best model
print("\n--- Tuned Model Classification Report ---")
print(classification_report(y_test_tuned, y_pred_tuned))
print("Tuned Model Accuracy:", accuracy_score(y_test_tuned, y_pred_tuned))

Fitting 5 folds for each of 8 candidates, totalling 40 fits

Best parameters found: {'C': 0.1, 'solver': 'saga'}
Best cross-validation accuracy: 0.8903804752107376

--- Tuned Model Classification Report ---
              precision    recall  f1-score   support

  Depression       0.89      0.89      0.89      3772
      Normal       0.90      0.89      0.89      3724
    Suicidal       0.88      0.89      0.89      3753

    accuracy                           0.89     11249
   macro avg       0.89      0.89      0.89     11249
weighted avg       0.89      0.89      0.89     11249

Tuned Model Accuracy: 0.8891457018401636


In [ ]:
import pandas as pd
import os
from datetime import datetime
import smtplib
import getpass

# Install googletrans library
!pip install googletrans==4.0.0-rc1

# Translation
from googletrans import Translator
translator = Translator()

# ------------------ EMAIL INPUT (ONCE) ------------------
EMAIL_SENDER = input("Enter Gmail (only once): ")
EMAIL_PASSWORD = getpass.getpass("Enter App Password (only once): ")

SMTP_SERVER = "smtp.gmail.com"
SMTP_PORT = 587

# ------------------ FAMILY LIST ------------------
family = ["father", "mother", "friend", "spouse"]

family_emails = {
    "father": "father@example.com",
    "mother": "mother@example.com",
    "friend": "friend@example.com",
    "spouse": "spouse@example.com"
}

# ------------------ TRANSLATION FUNCTION ------------------
def translate_to_english(text):
    try:
        translated = translator.translate(text, dest='en')
        return translated.text
    except:
        return text  # fallback if translation fails

# ------------------ EMAIL FUNCTION ------------------
def send_email(message):
    try:
        server = smtplib.SMTP(SMTP_SERVER, SMTP_PORT)
        server.starttls()
        server.login(EMAIL_SENDER, EMAIL_PASSWORD)

        for member in family:
            email = family_emails.get(member)
            if email:
                server.sendmail(EMAIL_SENDER, email, message)
                print(f" Email sent to {member}")

        server.quit()

    except Exception as e:
        print(" Email Error:", e)

# ------------------ ADVANCED SUPPORT ------------------
def advanced_support(risk_score, threshold=2):
    if risk_score >= threshold:
        print("\nAdvanced Support Recommended:")
        print("Consult therapist / counselor")
        return True
    return False

# ------------------ FAMILY CONNECTOR ------------------
def family_connector(risk_score, threshold=2, user_text=""):
    if risk_score >= threshold:

        print("\n AI Family Connector Activated")

        gentle_msg = "Your beloved one is in some trouble, please take care of him."

        print("\n Message to Family:")
        print(f" \"{gentle_msg}\"")

        full_message = f"""Subject: Alert

{gentle_msg}

User Message:
{user_text}
"""

        send_email(full_message)

        return True

    return False

# ------------------ STORE RESULTS ------------------
def store_results(user_id, text, sentiment, risk, ml_pred, interventions):
    result = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "user_id": user_id,
        "text": text,
        "sentiment": sentiment,
        "risk_score": risk,
        "ml_prediction": ml_pred,
        "interventions": interventions
    }

    df = pd.DataFrame([result])
    file_exists = os.path.exists("user_history.csv")
    df.to_csv("user_history.csv", mode='a', header=not file_exists, index=False)

# ------------------ DUMMY MODEL ------------------
def predict_post(text):
    risk = 3 if "help" in text.lower() else 1
    sentiment = "negative" if risk > 1 else "positive"
    ml_prediction = "high_risk" if risk > 1 else "low_risk"

    return {
        "text": text,
        "rule_based_sentiment": sentiment,
        "rule_based_risk": risk,
        "ml_prediction": ml_prediction
    }

# ------------------ MAIN LOOP ------------------
user_counter = 1

while True:
    user_text = input("\nEnter a post (any language, or 'exit'): ")

    if user_text.lower() == "exit":
       print("Goodbye! 😊👋")
       break # Corrected indentation

    # 🔥 STEP 1: TRANSLATE
    english_text = translate_to_english(user_text)

    print("\n Translated Text:", english_text)

    # 🔥 STEP 2: MODEL USES ENGLISH TEXT
    result = predict_post(english_text)

    print("\n--- Analysis ---")
    print("Original :", user_text)
    print("English  :", english_text)
    print("Sentiment:", result["rule_based_sentiment"])
    print("Risk     :", result["rule_based_risk"])

    escalation = advanced_support(result["rule_based_risk"])

    family_flag = family_connector(
        result["rule_based_risk"],
        user_text=english_text
    )

    interventions = []

    if escalation:
        interventions.append("Therapist Consultation")

    if family_flag:
        interventions.append("Family Alert Sent")

    print("\nInterventions:", interventions)

    store_results(
        user_id=user_counter,
        text=english_text,
        sentiment=result["rule_based_sentiment"],
        risk=result["rule_based_risk"],
        ml_pred=result["ml_prediction"],
        interventions="; ".join(interventions)
    )

    user_counter += 1

Enter Gmail (only once): sarkarshipra236@gmail.com
Enter App Password (only once): ··········

Enter a post (any language, or 'exit'): আমি খুব দুঃখিত

 Translated Text: I am very sorry

--- Analysis ---
Original : আমি খুব দুঃখিত
English  : I am very sorry
Sentiment: positive
Risk     : 1

Interventions: []

Enter a post (any language, or 'exit'): exit
Goodbye! 😊👋
